# 03b: End-to-End Timeseries (with HDF5 I/O)

The full SEAMLESS chain via the high-level `SeamlessPipeline`:
`parameterize → flow → kinematics`, saving the **UV projection** and the **flow** to
HDF5 and reloading both.

In [ ]:
import numpy as np
import h5py
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from seamless import SeamlessPipeline, FlowEstimator
from seamless.config import ParameterizationConfig, ThreeDNativeConfig, KinematicsConfig
from seamless.utils.loading import load_projections_complete, load_timepoint_data

## Load the sample timepoints

In [ ]:
with h5py.File('data/synthetic/ellipsoid.h5', 'r') as f:
    raw = f['microscopy_mockup'][:]                  # 2 timepoints (256³ each)
vols = [np.ascontiguousarray(raw[t]) for t in range(raw.shape[0])]
print('frames:', len(vols), '| volume shape:', vols[0].shape)

## Run the pipeline and save projection + flow

`SeamlessPipeline.run()` parameterizes every frame (warm-started), estimates the flow,
computes kinematics, and writes both HDF5 files. This is the compute-heavy cell.

In [ ]:
Path('outputs').mkdir(exist_ok=True)
pipe = SeamlessPipeline(
    vols, topology='cylinder', method='3d_native',
    threshold=43, num_target_points=8000, uv_res=128,
    param_config=ParameterizationConfig(iterations_t0=120, iterations_warm_start=40),
    three_d_config=ThreeDNativeConfig(num_iters=100, pe_degree=0),
    kinematics_config=KinematicsConfig(lagrangian_grid_res=48),
)
pipe.run(projection_h5='outputs/projection.h5',
         flow_h5='outputs/flow_results.h5', warm_iters=120)
print('frames:', len(pipe.frames), '| flow fields:', len(pipe.flow_fields))

## Reload the saved **projection** (save + import check)

The projection HDF5 follows the legacy `_uv_projection.h5` schema, so a fresh
`FlowEstimator` can be built straight from it.

In [ ]:
max_projs, xyz_vox, xyz_norm, pts_std, pts_mean, nuvo = \
    load_projections_complete('outputs/projection.h5', torch.device('cpu'))
print('projection timepoints:', sorted(max_projs), '| max_projection', max_projs[0].shape)

est = FlowEstimator.from_projection_h5('outputs/projection.h5')
print('reloaded frames:', len(est.frames),
      '| NuvoMLP restored:', est.frames[0].nuvo_model is not None)

## Reload the saved **flow** (save + import check)

In [ ]:
model, flow, xyz, kin = load_timepoint_data('outputs/flow_results.h5', 0,
                                            torch.device('cpu'), pe_degree=0)
print('flow:', tuple(flow.shape), '| surface pts:', tuple(xyz.shape))
print('kinematics keys:', sorted(kin), '| FlowMLP restored:', model is not None)

## Visualize

In [ ]:
eul = pipe.kinematics.compute_eulerian(pipe.flow_fields[0])
lag = pipe.kinematics.compute_lagrangian()
last_t = max(lag)
strain = lag[last_t]['analytical']['strain']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(pipe.frames[0].max_projection, cmap='gray', origin='lower')
axes[0].set_title('max projection (t0)')
im1 = axes[1].imshow(eul['divergence'], cmap='RdBu_r', origin='lower')
axes[1].set_title('divergence (t0->t1)')
im2 = axes[2].imshow(strain, cmap='viridis', origin='lower')
axes[2].set_title(f'cumulative strain (t={last_t})')
for a in axes: a.axis('off')
fig.colorbar(im1, ax=axes[1]); fig.colorbar(im2, ax=axes[2])
plt.tight_layout(); plt.show()

## Summary

`SeamlessPipeline` ties the whole API together and persists results: the **projection**
(`save_projection_h5` / `from_projection_h5`) and the **flow + kinematics**
(`KinematicsAnalyzer.save` / `load_timepoint_data`), both following the established
HDF5 schema so results round-trip cleanly.